# Comparative Discussion

This notebook brings together the findings from the image-based and structured fracture-data analyses conducted in the study. The SDNET2018 dataset was used to investigate concrete crack images through deep learning, segmentation, and crack morphology analysis, while the ConcreteTBF dataset was used to investigate fracture-energy behaviour through exploratory analysis and machine-learning regression.

The two datasets were analysed independently because they do not contain specimen-level correspondence. The comparative analysis therefore focuses on the complementary information provided by observable crack characteristics and quantitative fracture behaviour rather than attempting to establish a direct image-to-fracture-property relationship.

## 1. Environment Setup

This section defines the project paths and imports the libraries required to load and compare the results generated from the image-based and fracture-data analyses.

In [3]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.13.0+cpu
Device: cpu


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_METRICS = PROJECT_ROOT / "results" / "metrics"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"

print("Project root:", PROJECT_ROOT)
print("Metrics:", RESULTS_METRICS)
print("Tables:", RESULTS_TABLES)

Project root: c:\Users\yoolm\Documents\concrete-crack-and-fracture-analysis
Metrics: c:\Users\yoolm\Documents\concrete-crack-and-fracture-analysis\results\metrics
Tables: c:\Users\yoolm\Documents\concrete-crack-and-fracture-analysis\results\tables


## 2. Load Analysis Results

This section loads the principal results generated from the image-based and fracture-data analyses. The loaded results include CNN classification performance, crack morphology statistics, and machine-learning performance for fracture-energy prediction. These outputs form the basis for the comparative interpretation presented in this notebook.

In [4]:
# Load CNN results
cnn_performance = pd.read_csv(
    RESULTS_METRICS / "cnn_test_performance.csv"
)

cnn_report = pd.read_csv(
    RESULTS_METRICS / "cnn_classification_report.csv"
)

cnn_roc_auc = pd.read_csv(
    RESULTS_METRICS / "cnn_roc_auc.csv"
)

cnn_average_precision = pd.read_csv(
    RESULTS_METRICS / "cnn_average_precision.csv"
)

# Load morphology results
morphology_summary = pd.read_csv(
    RESULTS_TABLES / "morphology_comparison_summary.csv"
)

morphology_category = pd.read_csv(
    RESULTS_TABLES / "morphology_statistics_by_category.csv"
)

# Load fracture-analysis results
fracture_model_results = pd.read_csv(
    RESULTS_METRICS / "fracture_model_comparison.csv"
)

fracture_feature_importance = pd.read_csv(
    RESULTS_METRICS / "fracture_feature_importance.csv"
)

fracture_cv_results = pd.read_csv(
    RESULTS_METRICS / "fracture_cross_validation_results.csv"
)

fracture_predictions = pd.read_csv(
    RESULTS_METRICS / "fracture_test_predictions_residuals.csv"
)

print("CNN performance:", cnn_performance.shape)
print("CNN report:", cnn_report.shape)
print("Morphology summary:", morphology_summary.shape)
print("Morphology category:", morphology_category.shape)
print("Fracture model results:", fracture_model_results.shape)
print("Fracture feature importance:", fracture_feature_importance.shape)
print("Fracture CV results:", fracture_cv_results.shape)
print("Fracture predictions:", fracture_predictions.shape)

CNN performance: (4, 2)
CNN report: (5, 5)
Morphology summary: (5, 21)
Morphology category: (15, 10)
Fracture model results: (4, 7)
Fracture feature importance: (6, 4)
Fracture CV results: (4, 4)
Fracture predictions: (66, 9)


### 2.1 CNN Results

The finalized CNN evaluation results were loaded from the saved metric files generated during the CNN evaluation. These results include overall test-set performance, class-specific classification metrics, ROC-AUC, and average precision.

In [8]:
cnn_performance = pd.read_csv(
    RESULTS_METRICS / "cnn_test_performance.csv"
)

cnn_report = pd.read_csv(
    RESULTS_METRICS / "cnn_classification_report.csv"
)

cnn_roc_auc = pd.read_csv(
    RESULTS_METRICS / "cnn_roc_auc.csv"
)

cnn_average_precision = pd.read_csv(
    RESULTS_METRICS / "cnn_average_precision.csv"
)

print("CNN test performance:")
display(cnn_performance)

print("\nCNN classification report:")
display(cnn_report)

print("\nCNN ROC-AUC:")
display(cnn_roc_auc)

print("\nCNN average precision:")
display(cnn_average_precision)

CNN test performance:


,metric,value
0,accuracy,0.836701
1,precision,0.472162
2,recall,0.680031
3,f1_score,0.557345



CNN classification report:


,Unnamed: 0,precision,recall,f1-score,support
0,Uncracked,0.938165,0.864604,0.899883,7142.000000
1,Cracked,0.472162,0.680031,0.557345,1272.000000
2,accuracy,0.836701,0.836701,0.836701,0.836701
3,macro avg,0.705163,0.772318,0.728614,8414.000000
4,weighted avg,0.867716,0.836701,0.848100,8414.000000



CNN ROC-AUC:


,metric,value
0,roc_auc,0.855258



CNN average precision:


,metric,value
0,average_precision,0.661016


### 2.2 Morphology Results

The crack morphology results generated from the U-Net segmentation analysis were loaded from the saved summary tables. The loaded results include overall crack morphology characteristics and category-level statistics for deck, pavement, and wall images.

In [9]:
morphology_summary = pd.read_csv(
    RESULTS_TABLES / "morphology_comparison_summary.csv"
)

morphology_category = pd.read_csv(
    RESULTS_TABLES / "morphology_statistics_by_category.csv"
)

print("Overall morphology results:")
display(morphology_summary)

print("\nMorphology results by category:")
display(morphology_category)

Overall morphology results:


,Unnamed: 0,crack_area_percentage,crack_area_percentage.1,crack_area_percentage.2,crack_area_percentage.3,crack_length_pixels,crack_length_pixels.1,crack_length_pixels.2,crack_length_pixels.3,mean_width_pixels,...,mean_width_pixels.2,mean_width_pixels.3,max_width_pixels,max_width_pixels.1,max_width_pixels.2,max_width_pixels.3,branching_points,branching_points.1,branching_points.2,branching_points.3
0,NaN,count,mean,median,std,count,mean,median,std,count,...,median,std,count,mean,median,std,count,mean,median,std
1,category,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,deck,595,2.339,1.801,1.981,595,194.405,178.409,128.948,595,...,4.985,1.859,595,12.896,11.662,6.729,595,2.297,0.0,3.745
3,pavement,1249,4.292,3.413,3.955,1249,244.615,228.593,187.48,1249,...,6.525,3.061,1249,21.533,18.974,11.037,1249,5.696,2.0,8.546
4,wall,757,1.574,0.836,1.572,757,205.001,211.0,122.48,757,...,4.421,1.609,757,8.496,10.0,5.79,757,1.055,0.0,2.511



Morphology results by category:


,Category,Morphology,N,Mean,Median,Std,Minimum,Q1,Q3,Maximum
0,deck,Predicted crack area (%),595,2.339,1.801,1.981,0.031,0.769,3.638,17.085
1,deck,Predicted crack length (pixels),595,194.405,178.409,128.948,2.828,82.234,295.779,670.465
2,deck,Predicted mean width (pixels),595,5.177,4.985,1.859,2.000,4.494,5.609,20.043
3,deck,Predicted maximum width (pixels),595,12.896,11.662,6.729,2.000,10.000,14.560,57.585
4,deck,Predicted branching points,595,2.297,0.000,3.745,0.000,0.000,4.000,18.000
5,pavement,Predicted crack area (%),1249,4.292,3.413,3.955,0.031,1.169,6.178,26.135
6,pavement,Predicted crack length (pixels),1249,244.615,228.593,187.480,2.414,84.012,339.622,1023.235
7,pavement,Predicted mean width (pixels),1249,7.435,6.525,3.061,2.000,5.376,9.020,23.417
8,pavement,Predicted maximum width (pixels),1249,21.533,18.974,11.037,2.000,13.416,28.000,72.801
9,pavement,Predicted branching points,1249,5.696,2.000,8.546,0.000,0.000,8.000,66.000


### 2.3 Fracture-Data Results

The saved results from the ConcreteTBF fracture-energy analysis were loaded for comparative interpretation. These results include regression-model performance, five-fold cross-validation results, feature importance, and observation-level test predictions and residuals.

In [10]:
fracture_model_results = pd.read_csv(
    RESULTS_METRICS / "fracture_model_comparison.csv"
)

fracture_feature_importance = pd.read_csv(
    RESULTS_METRICS / "fracture_feature_importance.csv"
)

fracture_cv_results = pd.read_csv(
    RESULTS_METRICS / "fracture_cross_validation_results.csv"
)

fracture_predictions = pd.read_csv(
    RESULTS_METRICS / "fracture_test_predictions_residuals.csv"
)

print("Fracture model comparison:")
display(fracture_model_results)

print("\nFracture cross-validation results:")
display(fracture_cv_results)

print("\nFracture feature importance:")
display(fracture_feature_importance)

print("\nFracture test predictions and residuals:")
display(fracture_predictions.head())

Fracture model comparison:


,Model,CV_R2,CV_RMSE,CV_MAE,Test_R2,Test_RMSE,Test_MAE
0,Linear Regression,0.077094,0.026147,0.021004,0.124540,0.025899,0.020511
1,Random Forest,0.438277,0.020436,0.014666,0.604793,0.017401,0.012657
2,Gradient Boosting,0.380205,0.021482,0.015757,0.539562,0.018783,0.012431
3,XGBoost,0.425616,0.020702,0.015492,0.549163,0.018586,0.013757



Fracture cross-validation results:


,Model,CV_R2,CV_RMSE,CV_MAE
0,Linear Regression,0.077094,0.026147,0.021004
1,Random Forest,0.438277,0.020436,0.014666
2,Gradient Boosting,0.380205,0.021482,0.015757
3,XGBoost,0.425616,0.020702,0.015492



Fracture feature importance:


,Feature,Random Forest,Gradient Boosting,XGBoost
0,d (mm),0.136178,0.125462,0.152734
1,S (mm),0.093539,0.110137,0.131729
2,bw (mm),0.135109,0.172279,0.252971
3,a0 (mm),0.161598,0.123643,0.116181
4,ft (MPa),0.321375,0.320758,0.189699
5,E (GPa),0.152201,0.147721,0.156685



Fracture test predictions and residuals:


,Observed_Gc,Linear_Regression,Random_Forest,Gradient_Boosting,XGBoost,Linear_Regression_Residual,Random_Forest_Residual,Gradient_Boosting_Residual,XGBoost_Residual
0,0.0680,0.068262,0.067716,0.065096,0.071181,-0.000262,0.000284,0.002904,-0.003181
1,0.1100,0.055244,0.096396,0.098391,0.103155,0.054756,0.013604,0.011609,0.006845
2,0.0421,0.056382,0.047280,0.052182,0.048761,-0.014282,-0.005180,-0.010082,-0.006661
3,0.0320,0.067523,0.052521,0.056610,0.047819,-0.035523,-0.020521,-0.024610,-0.015819
4,0.0680,0.072112,0.064499,0.063420,0.055575,-0.004112,0.003501,0.004580,0.012425


## 3. Image-Based Analysis Findings

### 3.1 CNN Classification Performance

The CNN classification results were examined using the independent test set. Overall classification performance was assessed using accuracy, precision, recall, F1-score, ROC-AUC, and average precision, while class-specific performance was examined using the classification report.

In [15]:
cnn_metrics = {
    "Accuracy": cnn_performance.loc[
        cnn_performance["metric"] == "accuracy", "value"
    ].iloc[0],

    "Precision": cnn_performance.loc[
        cnn_performance["metric"] == "precision", "value"
    ].iloc[0],

    "Recall": cnn_performance.loc[
        cnn_performance["metric"] == "recall", "value"
    ].iloc[0],

    "F1-score": cnn_performance.loc[
        cnn_performance["metric"] == "f1_score", "value"
    ].iloc[0],

    "ROC-AUC": cnn_roc_auc.iloc[0, 1],

    "Average Precision": cnn_average_precision.iloc[0, 1]
}

cnn_metrics

{'Accuracy': np.float64(0.8367007368671262),
 'Precision': np.float64(0.4721615720524017),
 'Recall': np.float64(0.6800314465408805),
 'F1-score': np.float64(0.5573453608247423),
 'ROC-AUC': np.float64(0.8552582913723231),
 'Average Precision': np.float64(0.6610164064210885)}

The CNN achieved an accuracy of 83.67% on the independent test set, with a precision of 47.22%, recall of 68.00%, and F1-score of 55.73%. The ROC-AUC of 0.8553 indicates that the model demonstrated good discrimination between cracked and uncracked images across classification thresholds, while the average precision of 0.6610 provides additional information on its performance for the cracked class.

The class-specific results showed stronger performance for uncracked images than for cracked images. This indicates that, although the CNN was able to identify a substantial proportion of cracked images, distinguishing cracked images from uncracked images remained more challenging. Therefore, accuracy alone does not fully represent the classification performance, and the class-specific metrics and threshold-independent measures provide additional context.

In [16]:
cnn_report

,Unnamed: 0,precision,recall,f1-score,support
0,Uncracked,0.938165,0.864604,0.899883,7142.000000
1,Cracked,0.472162,0.680031,0.557345,1272.000000
2,accuracy,0.836701,0.836701,0.836701,0.836701
3,macro avg,0.705163,0.772318,0.728614,8414.000000
4,weighted avg,0.867716,0.836701,0.848100,8414.000000


### 3.2 U-Net Segmentation and Crack Morphology

A U-Net model was trained using the CrackSeg9k dataset for pixel-level crack segmentation and subsequently applied to a balanced sample of SDNET2018 images. The resulting predicted crack regions were used to extract image-based morphology descriptors, including crack area, crack length, crack width, orientation, and branching.

The segmentation results obtained on SDNET2018 are interpreted as predicted crack morphology rather than directly validated segmentation measurements because SDNET2018 does not provide native pixel-level crack masks. The transfer of the U-Net model from CrackSeg9k to SDNET2018 also introduces a domain-shift limitation that may affect segmentation quality.

### 3.2.1 U-Net performance

In [17]:
unet_history = pd.read_csv(
    RESULTS_METRICS / "unet_training_history.csv"
)

display(unet_history)

,epoch,train_loss,val_loss,train_dice,val_dice,train_iou,val_iou
0,1,1.054573,0.969749,0.222847,0.265985,0.170158,0.185803
1,2,0.913562,0.801136,0.324043,0.395538,0.228437,0.286984
2,3,0.733867,0.636675,0.472691,0.544514,0.355837,0.421560
3,4,0.593199,0.602083,0.590990,0.577866,0.470478,0.460815
4,5,0.536153,0.510762,0.629545,0.637822,0.508721,0.518185
5,6,0.496735,0.505252,0.653548,0.644739,0.533508,0.525894
6,7,0.471110,0.477348,0.669230,0.659444,0.549223,0.538034
7,8,0.457829,0.481402,0.675816,0.655776,0.556461,0.534709
8,9,0.453259,0.487543,0.680692,0.642382,0.561341,0.521006
9,10,0.429910,0.450862,0.693369,0.676311,0.574560,0.558842


In [18]:
best_unet_epoch = unet_history.loc[
    unet_history["val_dice"].idxmax()
]

print("Best validation epoch:", int(best_unet_epoch["epoch"]))
print("Best validation Dice:", best_unet_epoch["val_dice"])

Best validation epoch: 19
Best validation Dice: 0.7214837054843488


### 3.2.2 Predicted Crack Morphology

The predicted crack masks generated by the U-Net were used to quantify observable crack characteristics in the selected SDNET2018 sample. The analysis considered crack area, crack length, mean and maximum crack width, dominant orientation, and branching points.

Only images with non-zero predicted crack regions were included in the morphology summary. Consequently, the reported statistics describe the morphology of the predicted crack regions rather than the complete SDNET2018 image population.

In [19]:
morphology_category

,Category,Morphology,N,Mean,Median,Std,Minimum,Q1,Q3,Maximum
0,deck,Predicted crack area (%),595,2.339,1.801,1.981,0.031,0.769,3.638,17.085
1,deck,Predicted crack length (pixels),595,194.405,178.409,128.948,2.828,82.234,295.779,670.465
2,deck,Predicted mean width (pixels),595,5.177,4.985,1.859,2.000,4.494,5.609,20.043
3,deck,Predicted maximum width (pixels),595,12.896,11.662,6.729,2.000,10.000,14.560,57.585
4,deck,Predicted branching points,595,2.297,0.000,3.745,0.000,0.000,4.000,18.000
5,pavement,Predicted crack area (%),1249,4.292,3.413,3.955,0.031,1.169,6.178,26.135
6,pavement,Predicted crack length (pixels),1249,244.615,228.593,187.480,2.414,84.012,339.622,1023.235
7,pavement,Predicted mean width (pixels),1249,7.435,6.525,3.061,2.000,5.376,9.020,23.417
8,pavement,Predicted maximum width (pixels),1249,21.533,18.974,11.037,2.000,13.416,28.000,72.801
9,pavement,Predicted branching points,1249,5.696,2.000,8.546,0.000,0.000,8.000,66.000


In [20]:
print("Total morphology sample:", 6000)
print("Predicted positive masks:", 2601)
print("Predicted zero masks:", 6000 - 2601)
print("Predicted positive percentage:", 2601 / 6000 * 100)

Total morphology sample: 6000
Predicted positive masks: 2601
Predicted zero masks: 3399
Predicted positive percentage: 43.35


### 3.3 Morphology by Surface Category

The predicted crack morphology was examined separately for deck, pavement, and wall images to identify differences in the characteristics of the segmented crack regions. The comparison considers crack area, crack length, mean crack width, and branching behaviour across the three surface categories.

The category-level results are interpreted descriptively. Differences between categories may reflect both differences in the images and differences in segmentation performance, particularly because the U-Net model was trained on CrackSeg9k and applied to SDNET2018.

In [21]:
morphology_category

,Category,Morphology,N,Mean,Median,Std,Minimum,Q1,Q3,Maximum
0,deck,Predicted crack area (%),595,2.339,1.801,1.981,0.031,0.769,3.638,17.085
1,deck,Predicted crack length (pixels),595,194.405,178.409,128.948,2.828,82.234,295.779,670.465
2,deck,Predicted mean width (pixels),595,5.177,4.985,1.859,2.000,4.494,5.609,20.043
3,deck,Predicted maximum width (pixels),595,12.896,11.662,6.729,2.000,10.000,14.560,57.585
4,deck,Predicted branching points,595,2.297,0.000,3.745,0.000,0.000,4.000,18.000
5,pavement,Predicted crack area (%),1249,4.292,3.413,3.955,0.031,1.169,6.178,26.135
6,pavement,Predicted crack length (pixels),1249,244.615,228.593,187.480,2.414,84.012,339.622,1023.235
7,pavement,Predicted mean width (pixels),1249,7.435,6.525,3.061,2.000,5.376,9.020,23.417
8,pavement,Predicted maximum width (pixels),1249,21.533,18.974,11.037,2.000,13.416,28.000,72.801
9,pavement,Predicted branching points,1249,5.696,2.000,8.546,0.000,0.000,8.000,66.000


In [24]:
category_comparison = morphology_category.pivot(
    index="Category",
    columns="Morphology",
    values="Mean"
).reset_index()

category_comparison

Morphology,Category,Predicted branching points,Predicted crack area (%),Predicted crack length (pixels),Predicted maximum width (pixels),Predicted mean width (pixels)
0,deck,2.297,2.339,194.405,12.896,5.177
1,pavement,5.696,4.292,244.615,21.533,7.435
2,wall,1.055,1.574,205.001,8.496,3.837


In [25]:
print(morphology_category["Morphology"].unique())

<ArrowStringArray>
[        'Predicted crack area (%)',  'Predicted crack length (pixels)',
    'Predicted mean width (pixels)', 'Predicted maximum width (pixels)',
       'Predicted branching points']
Length: 5, dtype: str


### 3.3.1 Category Comparison

In [26]:
category_comparison = morphology_category.pivot(
    index="Category",
    columns="Morphology",
    values="Mean"
).reset_index()

display(category_comparison)

Morphology,Category,Predicted branching points,Predicted crack area (%),Predicted crack length (pixels),Predicted maximum width (pixels),Predicted mean width (pixels)
0,deck,2.297,2.339,194.405,12.896,5.177
1,pavement,5.696,4.292,244.615,21.533,7.435
2,wall,1.055,1.574,205.001,8.496,3.837


In [27]:
print(category_comparison.columns.tolist())

['Category', 'Predicted branching points', 'Predicted crack area (%)', 'Predicted crack length (pixels)', 'Predicted maximum width (pixels)', 'Predicted mean width (pixels)']


### 3.3.2 Interpretation of Category-Level Morphology

The category-level results show differences in the predicted crack morphology across deck, pavement, and wall images. Pavement images recorded the highest mean values for all five morphology measures. The mean predicted crack area was 4.292%, the mean crack length was 244.615 pixels, the mean crack width was 7.435 pixels, the mean maximum width was 21.533 pixels, and the mean number of branching points was 5.696.

Deck images recorded intermediate values for most morphology measures, with a mean predicted crack area of 2.339%, mean crack length of 194.405 pixels, mean width of 5.177 pixels, mean maximum width of 12.896 pixels, and mean branching points of 2.297. Wall images recorded the lowest mean predicted crack area (1.574%), mean width (3.837 pixels), maximum width (8.496 pixels), and branching points (1.055), while their mean predicted crack length was 205.001 pixels.

These differences indicate that the predicted crack regions varied across the three surface categories. However, the results should be interpreted descriptively rather than as evidence that one surface category inherently has more severe cracking. The morphology values are derived from U-Net predictions applied to SDNET2018, while the segmentation model was trained using CrackSeg9k. Differences may therefore reflect both the characteristics of the images and variations in segmentation performance caused by domain shift.

### 3.4 Summary of Image-Based Findings

The image-based analysis provided two complementary forms of information. The CNN evaluated whether individual SDNET2018 images were cracked or uncracked, while the U-Net provided predicted pixel-level crack regions from which observable morphology descriptors were extracted. The CNN achieved an accuracy of 83.67%, recall of 68.00% for the cracked class, F1-score of 55.73%, ROC-AUC of 0.8553, and average precision of 0.6610 on the independent test set.

The U-Net analysis further enabled quantitative description of predicted crack morphology, including crack area, length, width, orientation, and branching. Category-level analysis showed differences among deck, pavement, and wall images, with pavement images having the highest mean values for the evaluated morphology measures. These results demonstrate that image-based deep learning can provide both categorical information about crack presence and quantitative descriptors of predicted crack patterns.

However, the morphology results remain subject to segmentation uncertainty because SDNET2018 does not provide native pixel-level crack masks and the U-Net was trained on the separate CrackSeg9k dataset. Therefore, the extracted morphology descriptors are treated as predicted image-based characteristics rather than directly validated physical crack measurements.

## 4. Fracture-Data Analysis Findings

### 4.1 Fracture Energy

The ConcreteTBF dataset contains fracture energy (`Gc`) as the primary fracture-response variable considered in the machine-learning analysis. The 330 observations showed `Gc` values ranging from 0.010 to 0.141 N/mm, with a mean of 0.0664 N/mm and a median of 0.0650 N/mm.

The distribution of fracture energy was not uniform and showed variation across the observed range, with a concentration of observations in the lower-to-middle range and a smaller number of higher values. This variation provides a basis for examining whether the available geometric and material variables can explain or predict differences in fracture energy.

Because `Gc` is the fracture-energy parameter reported by the ConcreteTBF dataset, it was retained using the dataset's original terminology. It was not converted or relabelled as fracture toughness (`KIC`) because the available dataset information did not provide sufficient basis for a defensible conversion.

In [29]:
print("Fracture energy range:", 0.010, "to", 0.141, "N/mm")
print("Mean fracture energy:", 0.066410, "N/mm")
print("Median fracture energy:", 0.065000, "N/mm")

Fracture energy range: 0.01 to 0.141 N/mm
Mean fracture energy: 0.06641 N/mm
Median fracture energy: 0.065 N/mm


### 4.2 Variable Relationships

The relationships between fracture energy (`Gc`) and the available geometric and material variables were examined using Pearson correlation coefficients. The results showed generally weak linear relationships between `Gc` and the individual predictor variables. The correlations of `Gc` with beam depth (`d`), span length (`S`), beam width (`bw`), and initial notch depth (`a0`) were 0.0127, 0.1104, -0.0540, and 0.1155, respectively.

The material variables showed somewhat stronger negative relationships with fracture energy. The correlation between `Gc` and tensile strength (`ft`) was -0.2834, while the correlation between `Gc` and elastic modulus (`E`) was -0.2567. The correlation between `Gc` and ultimate load (`Pult`) was very weak at -0.0362.

In contrast, several strong relationships were observed among the predictor variables themselves. For example, beam depth and span length had a correlation of 0.8980, while tensile strength and elastic modulus had a correlation of 0.8280. Beam depth also showed strong relationships with beam width, notch depth, and ultimate load. These relationships indicate that the predictor variables are not independent of one another and should therefore be considered when interpreting machine-learning feature importance.

The weak individual correlations between `Gc` and most predictors also indicate that simple linear relationships may not adequately describe the variation in fracture energy. This provides a basis for examining nonlinear regression models in the subsequent analysis.

### 4.3 ML Prediction of Fracture Energy

Four regression models were evaluated for predicting the reported fracture energy (`Gc`): Linear Regression, Random Forest, Gradient Boosting, and XGBoost. Model performance was assessed using five-fold cross-validation and an independent 20% test set. The evaluation considered coefficient of determination (R²), root mean squared error (RMSE), and mean absolute error (MAE).

The Linear Regression model produced a cross-validation R² of 0.0771 and a test R² of 0.1245, indicating limited ability to represent the variation in fracture energy using a linear relationship. The nonlinear models produced higher test-set R² values. Random Forest achieved a test R² of 0.6048 with an RMSE of 0.0174 N/mm and an MAE of 0.0127 N/mm. Gradient Boosting achieved a test R² of 0.5396, while XGBoost achieved a test R² of 0.5492.

The cross-validation results also showed that the nonlinear models provided higher average R² values than Linear Regression. Random Forest recorded a cross-validation R² of 0.4383, followed by XGBoost at 0.4256 and Gradient Boosting at 0.3802, compared with 0.0771 for Linear Regression. These results indicate that nonlinear modelling captured more of the variation in the reported fracture-energy data than the linear baseline.

The differences between cross-validation and test-set performance also demonstrate that model performance depends on the particular data split. Therefore, the reported results are interpreted as evidence of predictive relationships within this dataset rather than as evidence of general performance on other concrete datasets.

In [30]:
display(fracture_model_results)

,Model,CV_R2,CV_RMSE,CV_MAE,Test_R2,Test_RMSE,Test_MAE
0,Linear Regression,0.077094,0.026147,0.021004,0.124540,0.025899,0.020511
1,Random Forest,0.438277,0.020436,0.014666,0.604793,0.017401,0.012657
2,Gradient Boosting,0.380205,0.021482,0.015757,0.539562,0.018783,0.012431
3,XGBoost,0.425616,0.020702,0.015492,0.549163,0.018586,0.013757


### 4.4 Feature Importance

Feature importance was examined for the three nonlinear regression models: Random Forest, Gradient Boosting, and XGBoost. The importance values indicate the relative contribution assigned to each predictor by the individual model and should not be interpreted as causal effects.

Tensile strength (`ft`) had the highest feature importance in both the Random Forest and Gradient Boosting models, with importance values of 0.3214 and 0.3208, respectively. In the XGBoost model, beam width (`bw`) had the highest importance at 0.2530, followed by tensile strength at 0.1897. Elastic modulus (`E`) showed moderate importance across the three models, while span length (`S`) generally had lower importance than the other predictors.

The differences in feature importance across the models indicate that the models relied on the available predictors in different ways when estimating fracture energy. The strong correlation between tensile strength and elastic modulus (`r = 0.8280`) also indicates that the importance assigned to these variables should not be interpreted as evidence of independent causal influence.

Overall, the feature-importance analysis identifies tensile strength, beam width, and elastic modulus among the more influential predictors within the evaluated models. These findings describe model behaviour within the ConcreteTBF dataset and do not establish physical causation.

In [31]:
display(fracture_feature_importance)

,Feature,Random Forest,Gradient Boosting,XGBoost
0,d (mm),0.136178,0.125462,0.152734
1,S (mm),0.093539,0.110137,0.131729
2,bw (mm),0.135109,0.172279,0.252971
3,a0 (mm),0.161598,0.123643,0.116181
4,ft (MPa),0.321375,0.320758,0.189699
5,E (GPa),0.152201,0.147721,0.156685


### 4.5 Prediction Errors

Prediction errors were examined using the absolute errors and residuals from the independent test set. The error analysis provides additional information beyond R², RMSE, and MAE by showing how closely the predicted fracture-energy values corresponded to the observed values.

The Linear Regression model generally produced larger absolute errors than the nonlinear models. The nonlinear models showed smaller typical errors, although differences remained among individual observations. The residual analysis also indicated that the nonlinear models produced residuals that were more concentrated around zero than the linear model.

Some larger prediction errors were observed at higher fracture-energy values. This indicates that the models did not predict all observations with the same level of accuracy across the response range. Consequently, the predictive performance should be interpreted together with the residual and absolute-error distributions rather than from a single performance metric.

The error analysis therefore provides evidence that the nonlinear models captured more of the variation in the reported fracture-energy data than the linear baseline, while also showing that prediction uncertainty remains within the dataset.

In [32]:
display(fracture_predictions.head())

,Observed_Gc,Linear_Regression,Random_Forest,Gradient_Boosting,XGBoost,Linear_Regression_Residual,Random_Forest_Residual,Gradient_Boosting_Residual,XGBoost_Residual
0,0.0680,0.068262,0.067716,0.065096,0.071181,-0.000262,0.000284,0.002904,-0.003181
1,0.1100,0.055244,0.096396,0.098391,0.103155,0.054756,0.013604,0.011609,0.006845
2,0.0421,0.056382,0.047280,0.052182,0.048761,-0.014282,-0.005180,-0.010082,-0.006661
3,0.0320,0.067523,0.052521,0.056610,0.047819,-0.035523,-0.020521,-0.024610,-0.015819
4,0.0680,0.072112,0.064499,0.063420,0.055575,-0.004112,0.003501,0.004580,0.012425


In [35]:
error_summary = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost"
    ],
    "Mean Absolute Error": [
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Linear_Regression"]).abs().mean(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Random_Forest"]).abs().mean(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Gradient_Boosting"]).abs().mean(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["XGBoost"]).abs().mean()
    ],
    "Median Absolute Error": [
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Linear_Regression"]).abs().median(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Random_Forest"]).abs().median(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Gradient_Boosting"]).abs().median(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["XGBoost"]).abs().median()
    ],
    "Maximum Absolute Error": [
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Linear_Regression"]).abs().max(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Random_Forest"]).abs().max(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["Gradient_Boosting"]).abs().max(),
        (fracture_predictions["Observed_Gc"] - fracture_predictions["XGBoost"]).abs().max()
    ]
})

display(error_summary)

,Model,Mean Absolute Error,Median Absolute Error,Maximum Absolute Error
0,Linear Regression,0.020511,0.017960,0.085755
1,Random Forest,0.012657,0.010083,0.082039
2,Gradient Boosting,0.012431,0.009784,0.098421
3,XGBoost,0.013757,0.011651,0.090986


## 5. Comparative Interpretation

### 5.1 Crack Characteristics and Fracture Behaviour

The image-based and fracture-data analyses provide complementary information about concrete cracking and fracture behaviour. The image-based component describes observable characteristics of cracking, including crack presence, predicted crack area, crack length, crack width, orientation, and branching. In contrast, the ConcreteTBF analysis describes quantitative fracture behaviour through the reported fracture energy (`Gc`) and its relationships with geometric and material variables.

The image-based analysis demonstrated that deep learning can distinguish cracked and uncracked concrete images and can also be used to generate predicted crack regions from which quantitative morphology descriptors can be extracted. The CNN achieved an accuracy of 83.67% and a ROC-AUC of 0.8553, while the U-Net-based analysis produced measurable differences in predicted crack morphology across the deck, pavement, and wall categories.

The fracture-data analysis showed that the available geometric and material variables had generally weak individual linear relationships with fracture energy. However, nonlinear regression models captured a greater proportion of the observed variation in `Gc` than the linear baseline. Random Forest, Gradient Boosting, and XGBoost achieved higher test-set R² values than Linear Regression, indicating that fracture-energy behaviour may involve nonlinear relationships among the available variables.

Taken together, the two analytical components demonstrate that observable crack characteristics and quantitative fracture behaviour can be investigated using complementary data-driven methods. However, the results do not establish a direct statistical relationship between the extracted image features and fracture energy because the SDNET2018 images and ConcreteTBF observations are independent datasets without specimen-level correspondence. The comparison therefore provides complementary insight into two aspects of concrete cracking and fracture behaviour rather than a direct image-to-fracture-energy prediction.

### 5.2 Image-Based Information vs Structured Fracture Information

The two analytical streams provide different forms of information about concrete behaviour. The image-based stream preserves visual information contained in concrete crack patterns. Through CNN classification and U-Net segmentation, this information can be represented using crack presence and predicted morphology descriptors such as area, length, width, orientation, and branching.

The structured fracture-data stream, in contrast, provides numerical information about specimen geometry, material properties, ultimate load, and fracture energy. The machine-learning analysis uses these variables to identify relationships and predict the reported fracture-energy response.

The distinction between the two information types is important. Image-based analysis provides information that can be observed from the surface of concrete, whereas the structured fracture dataset provides quantitative variables associated with fracture behaviour. Neither dataset alone represents the complete relationship between visible cracking and fracture response.

The complementary nature of the two streams therefore provides a broader analytical perspective. Image analysis can characterize observable crack patterns, while structured-data analysis can quantify relationships involving material, geometric, and fracture-related parameters. However, because the datasets are independent, the present study does not quantify a direct statistical relationship between individual crack morphology descriptors and individual fracture-energy measurements.

### 5.3 Deep Learning and Machine-Learning Complementarity

The analytical framework demonstrates the complementary roles of deep learning and machine learning in concrete crack and fracture analysis. The CNN was used to perform image-level classification of cracked and uncracked concrete surfaces, while the U-Net was used to obtain predicted pixel-level crack regions and derive quantitative morphology descriptors. These approaches operate directly on image data and therefore retain information about the visible characteristics of concrete cracking.

The machine-learning component addressed a different analytical task by using structured geometric and material variables to predict the reported fracture energy (`Gc`). The comparison of regression models showed that nonlinear models captured more of the variation in the fracture-energy data than the linear baseline. This demonstrates the usefulness of machine-learning methods for analysing relationships within structured fracture datasets.

The two approaches therefore serve different but complementary purposes within the proposed framework. Deep learning extracts and analyses information from digital images, whereas machine learning analyses relationships among numerical fracture-related variables. Their combined use provides a broader computational framework for investigating concrete cracking and fracture behaviour without requiring the two datasets to be directly combined.

The complementarity should not be interpreted as evidence that the image-based models predict fracture energy or fracture toughness. Establishing such a relationship would require a dataset in which crack images and corresponding fracture measurements are available for the same specimens. Such specimen-level correspondence is not present in the datasets used in this study.

### 5.4 Implications of the Combined Findings

The findings demonstrate the potential of data-driven methods to support the analysis of concrete cracking and fracture behaviour using different forms of publicly available data. The image-based analysis shows that deep learning can be used to classify concrete surfaces according to crack presence and to generate predicted crack regions from which observable morphology characteristics can be quantified.

The fracture-data analysis demonstrates that structured geometric and material variables can also be analysed using machine-learning regression models to investigate fracture-energy behaviour. The higher predictive performance of the nonlinear models compared with the linear baseline suggests that relationships within the ConcreteTBF dataset are not adequately represented by simple linear relationships alone.

From a methodological perspective, the findings support the use of complementary computational approaches when different types of concrete data are available. Image-based methods can provide quantitative descriptions of visible crack patterns, while structured-data methods can investigate numerical relationships associated with fracture response. This provides a framework that can be extended in future studies using datasets containing both digital crack images and corresponding mechanical or fracture measurements from the same specimens.

The present findings should nevertheless be interpreted within the scope of the secondary datasets used. In particular, the image-based morphology measurements are predicted quantities subject to segmentation uncertainty, while the fracture-energy models are evaluated only within the ConcreteTBF dataset. Therefore, the combined findings provide methodological and analytical insight rather than establishing a direct predictive relationship between crack morphology and fracture energy or fracture toughness.

## 6. Methodological Implications

The methodology demonstrates how publicly available datasets can be used to investigate different aspects of concrete cracking and fracture behaviour when direct laboratory testing is not available. The use of SDNET2018 enabled image-based analysis of concrete crack patterns, while the ConcreteTBF dataset provided structured numerical data for analysing fracture energy.

The results indicate that different computational methods are appropriate for different types of data. CNN-based classification was suitable for distinguishing cracked and uncracked images, while U-Net segmentation enabled the extraction of predicted crack morphology descriptors. For the structured fracture data, regression models provided a means of examining relationships between geometric and material variables and the reported fracture energy.

An important methodological implication is the need to distinguish between observable image characteristics and mechanically measured fracture properties. The image-based component provides predicted visual descriptors, whereas the ConcreteTBF component provides reported numerical fracture data. Treating these outputs as complementary rather than directly interchangeable helps maintain consistency with the information contained in each dataset.

The study also highlights the importance of dataset compatibility when attempting to integrate computer vision and fracture-mechanics analysis. A direct relationship between crack morphology and fracture response would require corresponding image and mechanical measurements from the same specimens. The independent datasets used in this study do not provide this correspondence. Future research can therefore build upon the present framework by developing paired datasets that combine digital crack images with experimentally measured fracture properties.

## 7. Limitations

Several limitations should be considered when interpreting the findings of this study.

First, the image-based and fracture-data components were based on independent public datasets. SDNET2018 contains labelled concrete crack images, while ConcreteTBF contains structured three-point-bending fracture data. The observations in the two datasets do not correspond to the same concrete specimens. Consequently, the study does not establish a direct specimen-level relationship between crack morphology and fracture energy or fracture toughness.

Second, SDNET2018 does not provide native pixel-level crack masks. Therefore, the U-Net segmentation model was trained using the separate CrackSeg9k dataset and subsequently applied to a selected SDNET2018 sample. The predicted morphology measurements are consequently subject to segmentation uncertainty and domain shift between the training and application datasets. The extracted morphology descriptors should therefore be interpreted as predicted image-based characteristics rather than directly validated physical crack measurements.

Third, the CNN classification performance was evaluated using the SDNET2018 test set, while the U-Net segmentation performance was evaluated on CrackSeg9k. The two models therefore have different evaluation datasets and performance measures. The U-Net predictions on SDNET2018 were not subjected to direct pixel-level accuracy assessment because corresponding ground-truth masks were unavailable.

Fourth, the fracture-data analysis was limited to the variables provided in the ConcreteTBF dataset. Although fracture energy (`Gc`) is a fracture-related parameter, it was not directly treated as fracture toughness (`KIC`). A reliable conversion would require additional theoretical and material information that could not be sufficiently established from the available dataset. Consequently, the machine-learning analysis focused on the reported fracture energy.

Fifth, the fracture-energy regression models were developed using a relatively small dataset of 330 observations. The models were evaluated using an 80/20 train-test split and five-fold cross-validation, but their performance should not be assumed to generalize to other concrete datasets without further validation.

Finally, the study relied entirely on secondary data and computational analysis because additional laboratory testing was not available. The findings therefore do not provide new experimental measurements of fracture toughness or direct experimental validation of the relationship between observed crack morphology and mechanical fracture behaviour.

## 8. Overall Synthesis

The study developed a data-driven framework for investigating concrete crack patterns and fracture behaviour using two complementary analytical streams based on publicly available datasets. The first stream used SDNET2018 digital images for deep learning-based crack classification and predicted crack morphology analysis, while the second used the ConcreteTBF dataset for structured analysis and machine-learning prediction of reported fracture energy.

The image-based analysis demonstrated that the CNN could distinguish cracked and uncracked concrete images, achieving 83.67% test accuracy and a ROC-AUC of 0.8553. The U-Net segmentation analysis further provided predicted crack regions from which quantitative descriptors such as crack area, length, width, orientation, and branching were extracted. Differences were observed in the predicted morphology across deck, pavement, and wall image categories, although these results remain subject to segmentation uncertainty and domain shift.

The fracture-data analysis showed that the available geometric and material variables had generally weak individual linear relationships with fracture energy. Nonlinear regression models captured more variation in the reported fracture-energy data than the Linear Regression baseline, with the evaluated nonlinear models producing higher test-set R² values. Feature-importance analysis further showed that the models relied on the available predictors differently, highlighting the model-dependent nature of the observed relationships.

Overall, the findings demonstrate that deep learning and machine learning can provide complementary approaches for analysing concrete cracking and fracture behaviour. Image-based methods provide quantitative information about observable crack characteristics, while structured-data models provide information about relationships involving geometric, material, and fracture-related variables. However, because the two datasets are independent, the study does not establish a direct image-to-fracture-energy or image-to-fracture-toughness relationship.

The study therefore provides a computational framework that can be extended in future research using paired datasets containing digital crack images and corresponding experimentally measured fracture properties. Such datasets would enable direct investigation of the relationship between observable crack morphology and quantitative fracture behaviour.